# CDT Paper - Figure 4: Attention vs Gradient Comparison

**Version**: v3.3.1 (dropout 0.3)
**Date**: 2025-12-23

This notebook generates:
- **Figure 4**: Comparison of top-20 positions from Attention vs Gradient analysis

Key finding: Attention and Gradient identify largely non-overlapping genomic regions

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import pearsonr
import torch
import torch.nn as nn
from dataclasses import dataclass
from typing import Optional, Dict, List
import h5py
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import shutil

# Paths
DRIVE_OUTPUT = Path("/content/drive/MyDrive/cdt_outputs/v3_3_1_dropout03")
DRIVE_DATA = Path("/content/drive/MyDrive/cdt_data")
FIGURE_OUTPUT = Path("/content/drive/MyDrive/cdt_outputs/paper_figures")
FIGURE_OUTPUT.mkdir(parents=True, exist_ok=True)

COLAB_DATA = Path("/content/colab_data_v3")
COLAB_DATA.mkdir(exist_ok=True)
TRAINING_DIR = COLAB_DATA / "training"
TRAINING_DIR.mkdir(exist_ok=True)

print(f"Output directory: {FIGURE_OUTPUT}")

In [ ]:
# Copy data to local
files_to_copy = [
    "human_proteomelm_embeddings_aligned.h5",
    "k562_gene_embeddings_aligned.h5",
    "protein_index_mapping_aligned.npz",
]

for f in files_to_copy:
    src = DRIVE_DATA / f
    dst = COLAB_DATA / f
    if src.exists() and not dst.exists():
        print(f"Copying {f}...")
        shutil.copy(src, dst)

val_src = DRIVE_DATA / "training" / "gasperini_val.h5"
val_dst = TRAINING_DIR / "gasperini_val.h5"
if val_src.exists() and not val_dst.exists():
    print("Copying gasperini_val.h5...")
    shutil.copy(val_src, val_dst)

print("Done!")

## 2. Model Definition

In [ ]:
@dataclass
class CDTv33Config:
    dna_dim: int = 3072
    dna_seq_len: int = 896
    protein_dim: int = 768
    rna_dim: int = 512
    n_proteins: int = 2360
    hidden_dim: int = 768
    nhead: int = 8
    dropout: float = 0.3
    dna_self_attn_layers: int = 2
    rna_self_attn_layers: int = 1
    protein_self_attn_layers: int = 1


class SequenceProjector(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, dropout: float = 0.1):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.norm(self.linear(x)))


class SelfAttentionBlock(nn.Module):
    def __init__(self, d_model: int, nhead: int = 4, dropout: float = 0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, return_attention=False):
        attn_out, attn_weights = self.self_attn(x, x, x, need_weights=return_attention, average_attn_weights=False)
        x = self.norm1(x + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        return x, attn_weights if return_attention else None


class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model: int, nhead: int = 4, dropout: float = 0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value):
        attn_out, attn_weights = self.cross_attn(query, key_value, key_value, need_weights=True, average_attn_weights=False)
        x = self.norm1(query + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        return x, attn_weights


class VirtualCellEmbedderWithAttention(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.dna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.rna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.protein_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.dna_attn = nn.MultiheadAttention(d_model, 4, dropout=dropout, batch_first=True)
        self.rna_attn = nn.MultiheadAttention(d_model, 4, dropout=dropout, batch_first=True)
        self.protein_attn = nn.MultiheadAttention(d_model, 4, dropout=dropout, batch_first=True)
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 3, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model)
        )

    def forward(self, dna, rna, protein):
        batch_size = dna.size(0)
        dna_pooled, _ = self.dna_attn(self.dna_query.expand(batch_size, -1, -1), dna, dna)
        rna_pooled, _ = self.rna_attn(self.rna_query.expand(batch_size, -1, -1), rna, rna)
        protein_pooled, _ = self.protein_attn(self.protein_query.expand(batch_size, -1, -1), protein, protein)
        concat = torch.cat([dna_pooled.squeeze(1), rna_pooled.squeeze(1), protein_pooled.squeeze(1)], dim=-1)
        return self.fusion(concat)


class CDTv33Model(nn.Module):
    def __init__(self, config: Optional[CDTv33Config] = None):
        super().__init__()
        if config is None:
            config = CDTv33Config()
        self.config = config
        
        self.dna_projector = SequenceProjector(config.dna_dim, config.hidden_dim, config.dropout)
        self.rna_projector = SequenceProjector(config.rna_dim, config.hidden_dim, config.dropout)
        self.protein_projector = SequenceProjector(config.protein_dim, config.hidden_dim, config.dropout)
        
        self.dna_self_attn_layers = nn.ModuleList([SelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout) for _ in range(config.dna_self_attn_layers)])
        self.rna_self_attn_layers = nn.ModuleList([SelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout) for _ in range(config.rna_self_attn_layers)])
        self.protein_self_attn_layers = nn.ModuleList([SelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout) for _ in range(config.protein_self_attn_layers)])
        
        self.dna_to_rna = CrossAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
        self.rna_to_protein = CrossAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
        
        self.vce = VirtualCellEmbedderWithAttention(config.hidden_dim, config.dropout)
        self.task_layer = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim), nn.GELU(), nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.n_proteins)
        )

    def forward(self, dna_emb, protein_emb, rna_emb, return_attention=False):
        batch_size = dna_emb.size(0)
        attention_maps = {}
        
        dna = self.dna_projector(dna_emb)
        rna = self.rna_projector(rna_emb)
        protein = self.protein_projector(protein_emb).unsqueeze(0).expand(batch_size, -1, -1)
        
        for layer in self.dna_self_attn_layers:
            dna, _ = layer(dna)
        for layer in self.rna_self_attn_layers:
            rna, _ = layer(rna)
        for layer in self.protein_self_attn_layers:
            protein, _ = layer(protein)
        
        rna_fused, dna_to_rna_attn = self.dna_to_rna(query=rna, key_value=dna)
        if return_attention:
            attention_maps['dna_to_rna'] = dna_to_rna_attn
        
        protein_fused, rna_to_protein_attn = self.rna_to_protein(query=protein, key_value=rna_fused)
        if return_attention:
            attention_maps['rna_to_protein'] = rna_to_protein_attn
        
        cell_embedding = self.vce(dna, rna_fused, protein_fused)
        logits = self.task_layer(cell_embedding)
        
        if return_attention:
            return logits, attention_maps
        return logits

print("Model classes defined.")

## 3. Dataset Definition

In [ ]:
class ValidationDataset(Dataset):
    def __init__(self, val_path, dna_path, protein_path, rna_path, mapping_path):
        data = np.load(mapping_path, allow_pickle=True)
        old_to_new_pairs = data['old_to_new']
        self.old_to_new_idx = {int(pair[0]): int(pair[1]) for pair in old_to_new_pairs}
        
        with h5py.File(val_path, 'r') as f:
            enformer_idx = f['enformer_idx'][:]
            orig_protein_idx = f['esm2_idx'][:]
            self.beta_values = f['beta'][:] if 'beta' in f else np.zeros_like(f['labels'][:], dtype=np.float32)
            self.enhancer_chr = [c.decode() if isinstance(c, bytes) else c for c in f['enhancer_chr'][:]]
            self.enhancer_start = f['enhancer_start'][:]
            self.enhancer_end = f['enhancer_end'][:]
        
        valid_mask = np.array([int(idx) in self.old_to_new_idx for idx in orig_protein_idx])
        self.beta_values = self.beta_values[valid_mask]
        self.enhancer_chr = [c for c, v in zip(self.enhancer_chr, valid_mask) if v]
        self.enhancer_start = self.enhancer_start[valid_mask]
        self.enhancer_end = self.enhancer_end[valid_mask]
        valid_orig_idx = orig_protein_idx[valid_mask]
        self.protein_idx = np.array([self.old_to_new_idx[int(idx)] for idx in valid_orig_idx])
        self.enhancer_centers = (self.enhancer_start + self.enhancer_end) // 2
        
        self.dna_file = h5py.File(dna_path, 'r')
        self.dna_emb = self.dna_file['embeddings']
        self.dna_centers = self.dna_file['centers'][:]
        self.dna_chroms = [c.decode() if isinstance(c, bytes) else c for c in self.dna_file['chroms'][:]]
        
        with h5py.File(protein_path, 'r') as f:
            self.protein_emb = f['embeddings'][:]
        
        with h5py.File(rna_path, 'r') as f:
            self.rna_emb = f['embeddings'][:]
        
        self._create_dna_mapping()
        print(f"Validation samples: {len(self.beta_values)}")
    
    def _create_dna_mapping(self):
        dna_coord_to_idx = {}
        for dna_idx, (chrom, center) in enumerate(zip(self.dna_chroms, self.dna_centers)):
            dna_coord_to_idx[(chrom, int(center))] = dna_idx
        
        self.sample_to_dna_idx = {}
        for sample_idx in range(len(self.enhancer_chr)):
            chrom = self.enhancer_chr[sample_idx]
            center = self.enhancer_centers[sample_idx]
            
            if (chrom, center) in dna_coord_to_idx:
                self.sample_to_dna_idx[sample_idx] = dna_coord_to_idx[(chrom, center)]
            else:
                found = False
                for dna_key, dna_idx in dna_coord_to_idx.items():
                    if dna_key[0] == chrom and abs(dna_key[1] - center) <= 1000:
                        self.sample_to_dna_idx[sample_idx] = dna_idx
                        found = True
                        break
                if not found:
                    self.sample_to_dna_idx[sample_idx] = None
    
    def __len__(self):
        return len(self.beta_values)
    
    def __getitem__(self, idx):
        dna_idx = self.sample_to_dna_idx.get(idx)
        if dna_idx is not None:
            dna_emb = self.dna_emb[dna_idx, :, :].astype(np.float32)
        else:
            dna_emb = np.zeros((896, 3072), dtype=np.float32)
        
        return {
            'dna_emb': torch.from_numpy(dna_emb),
            'rna_emb': torch.from_numpy(self.rna_emb.astype(np.float32)),
            'protein_idx': torch.tensor(int(self.protein_idx[idx]), dtype=torch.long),
            'beta': torch.tensor(self.beta_values[idx], dtype=torch.float32),
        }
    
    def get_protein_embeddings(self):
        return torch.from_numpy(self.protein_emb.astype(np.float32))
    
    def close(self):
        self.dna_file.close()

print("Dataset class defined.")

## 4. Load Model and Data

In [ ]:
val_dataset = ValidationDataset(
    val_path=TRAINING_DIR / "gasperini_val.h5",
    dna_path=DRIVE_DATA / "pilot_full_v2.h5",
    protein_path=COLAB_DATA / "human_proteomelm_embeddings_aligned.h5",
    rna_path=COLAB_DATA / "k562_gene_embeddings_aligned.h5",
    mapping_path=COLAB_DATA / "protein_index_mapping_aligned.npz"
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = CDTv33Model(CDTv33Config()).to(device)
model.load_state_dict(torch.load(DRIVE_OUTPUT / "cdt_v3_3_1_dropout03_best.pt", map_location=device))
model.eval()

protein_emb = val_dataset.get_protein_embeddings().to(device)
print(f"Model loaded. Protein embeddings: {protein_emb.shape}")

## 5. Extract Attention and Compute Gradients

In [ ]:
def get_top_k_positions(values, k=20):
    """Get indices of top-k positions by value"""
    return np.argsort(values)[-k:][::-1]

def compute_overlap(set1, set2):
    """Compute overlap between two sets of positions"""
    return len(set(set1) & set(set2))

print("Functions defined.")

In [ ]:
# Analyze samples
np.random.seed(42)
n_samples = min(100, len(val_dataset))
sample_indices = np.random.choice(len(val_dataset), n_samples, replace=False)

print(f"Analyzing {n_samples} samples...")

all_overlaps = []
all_attention_top20 = []
all_gradient_top20 = []

for i, idx in enumerate(tqdm(sample_indices, desc="Analyzing")):
    sample = val_dataset[idx]
    
    # Prepare input with gradient tracking
    dna = sample['dna_emb'].unsqueeze(0).to(device).requires_grad_(True)
    rna = sample['rna_emb'].unsqueeze(0).to(device)
    protein_idx = sample['protein_idx'].item()
    
    # Forward pass with attention
    logits, attention_maps = model(dna, protein_emb, rna, return_attention=True)
    
    # Get prediction for target gene
    pred = logits[0, protein_idx]
    
    # Compute gradient
    pred.backward()
    
    # Get DNA gradients (L2 norm across embedding dimension)
    dna_grad = dna.grad[0].detach().cpu().numpy()  # [896, 3072]
    gradient_importance = np.linalg.norm(dna_grad, axis=1)  # [896]
    
    # Get attention weights for target gene
    dna_to_rna_attn = attention_maps['dna_to_rna'][0].detach().cpu().numpy()  # [n_heads, n_genes, 896]
    attention_avg = dna_to_rna_attn.mean(axis=0)  # [n_genes, 896]
    gene_attention = attention_avg[protein_idx]  # [896]
    
    # Get top-20 positions
    attention_top20 = get_top_k_positions(gene_attention, k=20)
    gradient_top20 = get_top_k_positions(gradient_importance, k=20)
    
    # Compute overlap
    overlap = compute_overlap(attention_top20, gradient_top20)
    
    all_overlaps.append(overlap)
    all_attention_top20.append(attention_top20)
    all_gradient_top20.append(gradient_top20)
    
    # Clear gradients for next iteration
    model.zero_grad()

all_overlaps = np.array(all_overlaps)
overlap_percentage = all_overlaps / 20 * 100

print(f"\nResults:")
print(f"  Mean overlap: {all_overlaps.mean():.1f} / 20 positions ({overlap_percentage.mean():.1f}%)")
print(f"  Median overlap: {np.median(all_overlaps):.0f} / 20 positions")
print(f"  Min overlap: {all_overlaps.min()} / 20")
print(f"  Max overlap: {all_overlaps.max()} / 20")

## 6. Create Figure 4

In [ ]:
# Publication-quality figure
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 12

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=300)

# Panel A: Histogram of overlaps
ax1 = axes[0]
bins = np.arange(-0.5, 21.5, 1)
ax1.hist(all_overlaps, bins=bins, color='#2ecc71', edgecolor='white', alpha=0.8)
ax1.axvline(x=all_overlaps.mean(), color='red', linestyle='--', linewidth=2, 
            label=f'Mean: {all_overlaps.mean():.1f}')
ax1.set_xlabel('Number of Overlapping Positions (out of 20)', fontsize=12)
ax1.set_ylabel('Number of Samples', fontsize=12)
ax1.set_title('Attention vs Gradient: Top-20 Overlap', fontsize=14, fontweight='bold')
ax1.set_xlim(-0.5, 20.5)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# Add text annotation
stats_text = f'Mean overlap: {overlap_percentage.mean():.1f}%\nn = {len(all_overlaps)} samples'
ax1.text(0.97, 0.97, stats_text, transform=ax1.transAxes, fontsize=11,
         verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Panel B: Example comparison (single sample)
ax2 = axes[1]

# Pick a representative sample (median overlap)
median_idx = np.argsort(all_overlaps)[len(all_overlaps)//2]
example_attn = all_attention_top20[median_idx]
example_grad = all_gradient_top20[median_idx]

# Create visualization
positions = np.arange(896)
attn_mask = np.zeros(896)
grad_mask = np.zeros(896)
attn_mask[example_attn] = 1
grad_mask[example_grad] = 1

# Convert to genomic coordinates (kb from center)
pos_kb = (positions - 448) * 128 / 1000

ax2.fill_between(pos_kb, 0, attn_mask * 1.0, alpha=0.6, color='#3498db', label='Attention Top-20', step='mid')
ax2.fill_between(pos_kb, 0, -grad_mask * 1.0, alpha=0.6, color='#e74c3c', label='Gradient Top-20', step='mid')

ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.axvline(x=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)

ax2.set_xlabel('Distance from Enhancer Center (kb)', fontsize=12)
ax2.set_ylabel('', fontsize=12)
ax2.set_yticks([0.5, -0.5])
ax2.set_yticklabels(['Attention', 'Gradient'])
ax2.set_title('Example: Top-20 Positions (Single Sample)', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right', fontsize=10)
ax2.set_xlim(-60, 60)

# Add overlap count
overlap_count = all_overlaps[median_idx]
ax2.text(0.03, 0.03, f'Overlap: {overlap_count}/20', transform=ax2.transAxes, fontsize=11,
         verticalalignment='bottom', horizontalalignment='left',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Add panel labels
axes[0].text(-0.12, 1.05, 'A', transform=axes[0].transAxes, fontsize=18, fontweight='bold')
axes[1].text(-0.12, 1.05, 'B', transform=axes[1].transAxes, fontsize=18, fontweight='bold')

plt.tight_layout()

# Save
fig.savefig(FIGURE_OUTPUT / 'figure4_attention_vs_gradient.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURE_OUTPUT / 'figure4_attention_vs_gradient.pdf', bbox_inches='tight')
plt.show()

print(f"\nSaved to: {FIGURE_OUTPUT / 'figure4_attention_vs_gradient.png'}")

In [ ]:
# Cleanup
val_dataset.close()

print("\n" + "="*60)
print("Figure 4 Generation Complete!")
print("="*60)
print(f"\nKey findings:")
print(f"  - Mean overlap between Attention and Gradient top-20: {all_overlaps.mean():.1f} positions ({overlap_percentage.mean():.1f}%)")
print(f"  - This confirms that Attention and Gradient identify largely distinct regions")
print(f"\nOutput file: {FIGURE_OUTPUT / 'figure4_attention_vs_gradient.png'}")